# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² protein abundance dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # Do not use as a dict

# Print summary
print(f"{metadata.name}: {metadata.description}")
print(f"Published on: {getattr(metadata, 'datePublished', 'Unknown')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Keywords: {', '.join(getattr(metadata, 'keywords', []))}")

## 2. Data Overview
Review available record sets, fields, and their IDs (using `@id`).

In [ ]:
# Explore record sets
from pprint import pprint

print("Record Sets available in this dataset:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- @id: {rs.id}, Name: {getattr(rs, 'name', '(no name)')}, Description: {getattr(rs, 'description', '(no description)')}")
    # List fields for each record set
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}, type: {getattr(field, 'data_type', '')}")
    print("")
# For reference, show the first record of the first record set
if record_sets:
    sample_records = list(dataset.records(record_set=record_sets[0].id))
    print(f"First record of {record_sets[0].id}:")
    pprint(sample_records[0])
else:
    print("No record sets found!")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For this dataset, we use the main record set with `@id='https://api.app.sen.science/frontiers/7154140/d44810b3-b26d-44ea-b063-c3ea810c618c'` which typically contains the principal protein data table (check the cell above to verify actual `@id` from your schema).

In [ ]:
# Choose record set IDs to load (replace with your dataset's discovered IDs)
# You may need to adjust these if different record sets are present.
record_sets_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set: {record_set_id}")
    print(f"Fields: {df.columns.tolist()}")
    print(df.head(2), "\n")
# Pick the main data for further exploration (select one record_set_id)
main_record_set_id = record_sets_ids[0] if record_sets_ids else None
if main_record_set_id:
    print(f"Continuing with main record set: {main_record_set_id}")
    main_df = dataframes[main_record_set_id]
    print(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we'll analyze the protein abundance by filtering for proteins with high peptide count (e.g., `Peptide_Count` > 10) and normalize their molecular weight (`MW_kDa`). Be sure to match these field names/IDs to the actual column names in your DataFrame (print main_df.columns to check).

In [ ]:
# Show available fields in main_df
print("Fields in main_df:", main_df.columns.tolist())

# Choose field names (by @id, but in dataframe it's usually column names)
# Example: peptide_count_field = '@id:Peptide_Count' or the actual name used in your schema
#          mw_field = '@id:MW_kDa' or actual name

# Try to infer likely fields (adjust if needed)
peptide_count_field = None
mw_field = None
for col in main_df.columns:
    if 'peptide' in col.lower() and 'count' in col.lower():
        peptide_count_field = col
    if col.lower() in ['mw_kda', 'molecular_weight', 'mw', 'molecular_weight_kda'] or ('weight' in col.lower()):
        mw_field = col

if (peptide_count_field is None) or (mw_field is None):
    print("Could not automatically find peptide count or MW fields. Please assign the correct column names based on schema.")
    # Example fallback:
    # peptide_count_field = '<@id or column name of peptide count>'
    # mw_field = '<@id or column name of molecular weight>'
else:
    print(f"Using '{peptide_count_field}' as peptide count and '{mw_field}' as MW.")
    # Filter
    threshold = 10
    filtered_df = main_df[main_df[peptide_count_field] > threshold].copy()
    print(f"Filtered records with {peptide_count_field} > {threshold}: {len(filtered_df)} records.")
    print(filtered_df[[peptide_count_field, mw_field]].head())
    # Normalize molecular weight field
    filtered_df[f"{mw_field}_normalized"] = (
        filtered_df[mw_field] - filtered_df[mw_field].mean()
    ) / filtered_df[mw_field].std()
    print(f"Normalized {mw_field}:\n", filtered_df[[mw_field, f"{mw_field}_normalized"]].head())
    # Group by a key attribute if available (e.g., 'description' or 'Accession')
    group_field = None
    for col in main_df.columns:
        if 'description' in col.lower() or 'gene' in col.lower() or 'group' in col.lower():
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[mw_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean MW):")
        print(grouped_df.head())
    else:
        print("No suitable group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The following cell shows the distribution of molecular weight for proteins with a high peptide count.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if (peptide_count_field is not None) and (mw_field is not None) and (not filtered_df.empty):
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[mw_field], bins=30, kde=True)
    plt.xlabel(f"{mw_field}")
    plt.ylabel("# Proteins")
    plt.title(f"Distribution of {mw_field} for Proteins with {peptide_count_field} > {threshold}")
    plt.show()
else:
    print("Cannot plot: check that numerical analysis fields were properly identified.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² protein abundance dataset using the `mlcroissant` library.

* We inspected dataset record sets and their fields by unique `@id`.
* We extracted and explored protein data, filtered records based on peptide count, and normalized molecular weights.
* We visualized protein molecular weight distributions for high-abundance entries.

These steps can be further extended for advanced biomarker discovery or statistical studies. For more dataset details, visit the [dataset landing page](https://sen.science/doi/10.71728/senscience.vq4a-28xa) or review the Croissant schema JSON at the provided URL.